# 02. Introduction to Pandas for EDA & Machine Learning

Welcome to the second notebook in the **Intro to EDA** series!  
In `01_numpy.ipynb`, we saw how fast NumPy is with pure numbers. Real-world datasets, however, look like messy spreadsheets: they have **column names** (`"city"`, `"price"`), **text**, **dates**, **missing cells**, and **duplicate rows**.

**Pandas** sits right on top of NumPy and gives us labeled tables (**`DataFrame`** and **`Series`**) built specifically for **Exploratory Data Analysis (EDA)** and data cleaning.

---

### 📚 Table of Contents
1. [Introduction & Core Structures (`Series` & `DataFrame`)](#1.-Introduction-&-Core-Structures-(Series-&-DataFrame))
2. [Loading & Saving Data (`read_csv` & `to_csv`)](#2.-Loading-&-Saving-Data-(read_csv-&-to_csv))
3. [First-Look EDA: Inspecting Your Dataset (`info`, `describe`, `value_counts`)](#3.-First-Look-EDA:-Inspecting-Your-Dataset)
4. [Selecting, Indexing & Filtering (`loc`, `iloc`, Boolean Masks)](#4.-Selecting,-Indexing-&-Filtering)
5. [Data Cleaning Essentials (Missing Values, Duplicates, Types & Text)](#5.-Data-Cleaning-Essentials)
6. [Modifying Columns, Sorting & Simple Feature Engineering](#6.-Modifying-Columns,-Sorting-&-Simple-Feature-Engineering)
7. [Grouping & Aggregating (`groupby` & `crosstab`)](#7.-Grouping-&-Aggregating)
8. [Combining Tables (`concat` & `merge`)](#8.-Combining-Tables)
9. [Date & Time Essentials](#9.-Date-&-Time-Essentials)
10. [Capstone Mini-EDA Lab & Summary Cheat Sheet](#10.-Capstone-Mini-EDA-Lab-&-Summary-Cheat-Sheet)

---
## 1. Introduction & Core Structures (`Series` & `DataFrame`)

### 🌍 Real-World Use Case
Imagine you export a table of **customer orders** from a store database:
- One column has **text** (`"Cairo"`, `"Alexandria"`),
- One column has **numbers** (`299.99`, `45.0`),
- One column has **dates** (`"2026-03-10"`).

NumPy forces an entire array to share a single data type, which is awkward for mixed tables. **Pandas** solves this with two objects:
1. **`Series`**: A **1D labeled column** (like a single column in Excel with row labels/index).
2. **`DataFrame`**: A **2D labeled table** (a collection of `Series` sharing the same row index — just like a full spreadsheet or SQL table).

In [3]:
import numpy as np
import pandas as pd

print("Pandas version:", pd.__version__)

# 1) Series: a single labeled column (e.g., daily website visitors)
visitors = pd.Series([120, 145, 98, 210], index=["Mon", "Tue", "Wed", "Thu"], name="visitors")
print("Series example:\n", visitors)

# 2) DataFrame: created from a Python dictionary (keys = column names, values = lists)
df_customers = pd.DataFrame({
    "name": ["Sara", "Omar", "Lina", "Youssef"],
    "city": ["Cairo", "Alexandria", "Cairo", "Giza"],
    "age":  [28, 34, 22, 41],
    "total_spent": [450.0, 1200.5, 89.0, 620.0]
})

print("\nDataFrame example:")
df_customers

Pandas version: 3.0.3
Series example:
 Mon    120
Tue    145
Wed     98
Thu    210
Name: visitors, dtype: int64

DataFrame example:


,name,city,age,total_spent
0,Sara,Cairo,28,450.0
1,Omar,Alexandria,34,1200.5
2,Lina,Cairo,22,89.0
3,Youssef,Giza,41,620.0


---
## 2. Loading & Saving Data (`read_csv` & `to_csv`)

### 🌍 Real-World Use Case
In real EDA projects, you rarely type data by hand — you load datasets from **CSV files** (`pd.read_csv("orders.csv")`), clean them, and export the cleaned version (`df.to_csv("orders_clean.csv", index=False)`) for visualization or machine learning.

> **Note:** Below we use Python's `StringIO` to simulate reading a CSV file in memory so this notebook works out-of-the-box anywhere without needing external files. In your own projects, you simply pass the file path like `pd.read_csv("sales_data.csv")`.

In [4]:
from io import StringIO

# Simulated messy e-commerce CSV data
raw_csv = """order_id,customer,city,category,price,quantity,order_date,is_fraud
1001,Sara,Cairo,Electronics,500,1,2026-01-05,No
1002,Omar,Alexandria,Clothing,80,2,2026-01-06,No
1003,Lina,cairo ,Books,25,3,2026-01-06,No
1004,Youssef,Giza,Electronics,1200,1,2026-01-08,Yes
1005,Mona,Cairo,Clothing,,2,2026-01-09,No
1006,Karim,Alexandria,Books,30,4,2026-01-10,No
1006,Karim,Alexandria,Books,30,4,2026-01-10,No
1007,Nour,Cairo,Electronics,450,1,2026-01-11,No
1008,Tarek,Giza,Clothing,120,1,2026-01-12,No
1009,Hana,Cairo,Books,invalid,2,2026-01-14,No
1010,Ziad,Alexandria,Electronics,800,1,2026-01-15,No
"""

# In real life: df = pd.read_csv("orders.csv")
df = pd.read_csv(StringIO(raw_csv))
df.head()

,order_id,customer,city,category,price,quantity,order_date,is_fraud
0,1001,Sara,Cairo,Electronics,500,1,2026-01-05,No
1,1002,Omar,Alexandria,Clothing,80,2,2026-01-06,No
2,1003,Lina,cairo,Books,25,3,2026-01-06,No
3,1004,Youssef,Giza,Electronics,1200,1,2026-01-08,Yes
4,1005,Mona,Cairo,Clothing,NaN,2,2026-01-09,No


> **💡 Tip for Saving:** When saving a cleaned DataFrame with `df.to_csv("clean_orders.csv", index=False)`, always pass **`index=False`** — otherwise Pandas writes an extra `Unnamed: 0` column for row numbers!

---
## 3. First-Look EDA: Inspecting Your Dataset

Before cleaning or plotting anything, every data analyst runs a **60-second health check** on a new DataFrame.

### 3.1 `head()`, `tail()`, `sample()` and `shape`
#### 🌍 Real-World Use Case
If a CSV has 500,000 rows, you cannot print the whole table.
- **`head(5)` / `tail(5)`** verify the file loaded properly (checking headers and bottom rows for weird footers).
- **`sample(3)`** shows random rows so you don't fool yourself if the top rows happen to be sorted or unusually clean.

In [5]:
print("Shape (rows, columns):", df.shape)
print("Column names:", df.columns.tolist())

print("\nFirst 3 rows:")
display(df.head(3))

print("Random 3 rows:")
display(df.sample(3, random_state=42))

Shape (rows, columns): (11, 8)
Column names: ['order_id', 'customer', 'city', 'category', 'price', 'quantity', 'order_date', 'is_fraud']

First 3 rows:


,order_id,customer,city,category,price,quantity,order_date,is_fraud
0,1001,Sara,Cairo,Electronics,500,1,2026-01-05,No
1,1002,Omar,Alexandria,Clothing,80,2,2026-01-06,No
2,1003,Lina,cairo,Books,25,3,2026-01-06,No


Random 3 rows:


,order_id,customer,city,category,price,quantity,order_date,is_fraud
5,1006,Karim,Alexandria,Books,30,4,2026-01-10,No
0,1001,Sara,Cairo,Electronics,500,1,2026-01-05,No
9,1009,Hana,Cairo,Books,invalid,2,2026-01-14,No


### 3.2 `df.info()` and `df.describe()`

#### 🌍 Real-World Use Case & Insights
- **`df.info()`** tells you **two critical things immediately**:
  1. **Missing values:** Compare `Non-Null Count` against the total number of rows (`11`). Here, `price` only has `10` non-null values — **1 value is missing!**
  2. **Wrong data types:** Look at `price` — it says `object` (text) instead of `float64`! Why? Because row 1009 contains the word `"invalid"`. You just caught a hidden bug before doing any math!
- **`df.describe()`** gives summary statistics for numeric columns (`mean`, `std`, `min`, `25%`, `50%` median, `75%`, `max`). Comparing `mean` vs. `50%` helps you spot **skewed data or extreme outliers** (like a `$1,000,000` typo), and `min` catches impossible values (like negative ages or prices).

In [6]:
print("--- df.info() ---")
df.info()

print("\n--- df.describe() for numeric columns ---")
display(df.describe())

print("--- df.describe(include='object') for text columns ---")
display(df.describe(include="object"))

--- df.info() ---
<class 'pandas.DataFrame'>
RangeIndex: 11 entries, 0 to 10
Data columns (total 8 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   order_id    11 non-null     int64
 1   customer    11 non-null     str  
 2   city        11 non-null     str  
 3   category    11 non-null     str  
 4   price       10 non-null     str  
 5   quantity    11 non-null     int64
 6   order_date  11 non-null     str  
 7   is_fraud    11 non-null     str  
dtypes: int64(2), str(6)
memory usage: 1.2 KB

--- df.describe() for numeric columns ---


,order_id,quantity
count,11.000000,11.000000
mean,1005.545455,2.000000
std,2.876235,1.183216
min,1001.000000,1.000000
25%,1003.500000,1.000000
50%,1006.000000,2.000000
75%,1007.500000,2.500000
max,1010.000000,4.000000


--- df.describe(include='object') for text columns ---


C:\Users\Compu Tech\AppData\Local\Temp\ipykernel_5408\2893108928.py:8: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  display(df.describe(include="object"))


,customer,city,category,price,order_date,is_fraud
count,11,11,11,10,11,11
unique,10,4,3,9,9,2
top,Karim,Cairo,Electronics,30,2026-01-06,No
freq,2,4,4,2,2,10


### 3.3 `value_counts()` — Your Secret Weapon for Categorical EDA

#### 🌍 Real-World Use Case & Insights
Suppose you are building a **fraud detection** or **customer churn** machine learning model, and you run `df["is_fraud"].value_counts(normalize=True)`:
- **Insight 1 — Class Imbalance Check:** You immediately see `No: 90.9%` vs. `Yes: 9.1%`! In real fraud datasets, it's often `99.5% vs. 0.5%`. Knowing about **class imbalance** early is crucial because a naive ML model that predicts `"No"` for every order would look "99.5% accurate" while catching **zero** fraud!
- **Insight 2 — Spotting Dirty Categories & Typos:** Running `df["city"].value_counts()` reveals `"Cairo"` (4 times) and `"cairo "` (1 time, lowercase with a trailing space) as two separate cities! Without `value_counts()`, your city revenue report would split Cairo into two groups.

In [7]:
print("--- Fraud Class Balance (counts) ---")
print(df["is_fraud"].value_counts())

print("\n--- Fraud Class Balance (percentages) ---")
print(df["is_fraud"].value_counts(normalize=True).round(3) * 100)

print("\n--- City Counts (Notice dirty 'cairo '!) ---")
print(df["city"].value_counts())

--- Fraud Class Balance (counts) ---
is_fraud
No     10
Yes     1
Name: count, dtype: int64

--- Fraud Class Balance (percentages) ---
is_fraud
No     90.9
Yes     9.1
Name: proportion, dtype: float64

--- City Counts (Notice dirty 'cairo '!) ---
city
Cairo         4
Alexandria    4
Giza          2
cairo         1
Name: count, dtype: int64


---
## 4. Selecting, Indexing & Filtering

### 🌍 Real-World Use Case
Once you inspect your dataset, you rarely analyze all columns and rows at once:
- You want to inspect **only** the `customer` and `city` columns.
- Or pull out **only** the `Electronics` orders in `Cairo` to investigate a regional trend.
- Or isolate the suspicious `"invalid"` rows to see what went wrong.

### 4.1 Selecting Columns & Using `loc[]` vs. `iloc[]`
- `df["col"]` returns a **1D `Series`**; `df[["col1", "col2"]]` (double brackets) returns a **2D `DataFrame`**.
- **`df.loc[row_labels, col_names]`** selects by **label / name** (and condition masks).
- **`df.iloc[row_positions, col_positions]`** selects by **integer index position** (`0, 1, 2...`), just like NumPy slicing!

In [8]:
# Select multiple columns
print("Customer & City columns:")
display(df[["customer", "city"]].head(3))

# loc: rows with index 0 to 2 (inclusive!) and specific column names
print("Using .loc[0:2, ['customer', 'price']]:")
display(df.loc[0:2, ["customer", "price"]])

# iloc: first 3 rows (0, 1, 2 -> 3 excluded!) and first 4 columns
print("Using .iloc[0:3, 0:4]:")
display(df.iloc[0:3, 0:4])

Customer & City columns:


,customer,city
0,Sara,Cairo
1,Omar,Alexandria
2,Lina,cairo


Using .loc[0:2, ['customer', 'price']]:


,customer,price
0,Sara,500
1,Omar,80
2,Lina,25


Using .iloc[0:3, 0:4]:


,order_id,customer,city,category
0,1001,Sara,Cairo,Electronics
1,1002,Omar,Alexandria,Clothing
2,1003,Lina,cairo,Books


### 4.2 Filtering Rows with Conditions (Boolean Masks)
Just like in NumPy, combine multiple conditions using **`&`** (AND), **`|`** (OR), **`~`** (NOT), and always wrap each condition in parentheses `(...)`.  
Two super clean Pandas helpers:
- **`.isin([...])`**: Check if a value belongs to a list (e.g., `city` is Cairo *or* Giza).
- **`.between(low, high)`**: Check if a number falls inside a range.

In [9]:
# Orders from Cairo OR Giza with quantity >= 2
mask = df["city"].isin(["Cairo", "Giza"]) & (df["quantity"] >= 2)
print("Cairo/Giza orders with quantity >= 2:")
display(df.loc[mask, ["order_id", "customer", "city", "quantity"]])

# Orders with quantity between 2 and 4 (inclusive)
print("Quantity between 2 and 4:")
display(df[df["quantity"].between(2, 4)].head(4))

Cairo/Giza orders with quantity >= 2:


,order_id,customer,city,quantity
4,1005,Mona,Cairo,2
9,1009,Hana,Cairo,2


Quantity between 2 and 4:


,order_id,customer,city,category,price,quantity,order_date,is_fraud
1,1002,Omar,Alexandria,Clothing,80,2,2026-01-06,No
2,1003,Lina,cairo,Books,25,3,2026-01-06,No
4,1005,Mona,Cairo,Clothing,NaN,2,2026-01-09,No
5,1006,Karim,Alexandria,Books,30,4,2026-01-10,No


### ✏️ Practice Exercise 1: Inspection & Filtering
Using `df`:
1. Use `.value_counts()` on the `"category"` column to see how many orders belong to each product category.
2. Filter `df` to show only `"Electronics"` orders where `"is_fraud"` is `"No"`, displaying only the `order_id`, `customer`, and `city` columns.

In [10]:
# Use .value_counts() on the "category" column to see how many orders belong to each product category.
count = df["category"].value_counts()
print(count)

category
Electronics    4
Books          4
Clothing       3
Name: count, dtype: int64


In [11]:
# Filter df to show only "Electronics" orders where "is_fraud" is "No", displaying only the order_id, customer, and city columns.
new_df = df[(df['category'] == 'Electronics') & (df['is_fraud'] == 'No')]
res = new_df[['order_id', 'customer', 'city']]
print(res)

    order_id customer        city
0       1001     Sara       Cairo
7       1007     Nour       Cairo
10      1010     Ziad  Alexandria


<details>
<summary><strong>💡 Click to view solution</strong></summary>

```python
print(df["category"].value_counts())

safe_elec = df.loc[
    (df["category"] == "Electronics") & (df["is_fraud"] == "No"),
    ["order_id", "customer", "city"]
]
safe_elec
```

</details>

---
## 5. Data Cleaning Essentials

In real-world data analysis, **data cleaning takes ~70–80% of the work**. Let's fix every issue we spotted in `df.info()` and `value_counts()`!

### 🌍 Real-World Use Case (Why Each Step Matters)
1. **Clean messy strings (`.str.strip().str.title()`):** Merges `"cairo "` and `"Cairo"` so sales reports aren't fragmented.
2. **Remove duplicates (`drop_duplicates()`):** Prevents double-counting revenue if an order was accidentally logged twice (look at `order_id == 1006`!).
3. **Fix numeric columns (`pd.to_numeric(..., errors="coerce")`):** Converts `"invalid"` into `NaN` so the rest of the `price` column becomes proper `float64` numbers instead of crashing your math.
4. **Handle `NaN` missing values (`dropna()` vs. `fillna()`):** Blindly calling `dropna()` can throw away valuable rows if only one column is missing; imputing with the **median** (`fillna`) keeps your sample size intact while resisting outliers!

> **⚠️ Watch Out:** Always work on **`df_clean = df.copy()`** so you keep the original raw DataFrame untouched, and assign changes back (`df_clean["col"] = ...`) rather than using `inplace=True`.

In [12]:
df_clean = df.copy()

# 1. Clean whitespace & capitalization in text columns
df_clean["city"] = df_clean["city"].str.strip().str.title()
print("Cleaned cities:\n", df_clean["city"].value_counts())

# 2. Check and remove exact duplicate rows (order 1006 appeared twice!)
print("\nDuplicate rows found:", df_clean.duplicated().sum())
df_clean = df_clean.drop_duplicates().reset_index(drop=True)

# 3. Convert 'price' to numeric; unparseable strings ('invalid') become NaN
df_clean["price"] = pd.to_numeric(df_clean["price"], errors="coerce")

# 4. Inspect missing values, then impute missing prices with the median price
print("\nMissing values per column:\n", df_clean.isna().sum())
median_price = df_clean["price"].median()
df_clean["price"] = df_clean["price"].fillna(median_price)

print(f"\nFilled missing prices with median (${median_price}). Remaining NaNs:", df_clean["price"].isna().sum())
df_clean

Cleaned cities:
 city
Cairo         5
Alexandria    4
Giza          2
Name: count, dtype: int64

Duplicate rows found: 1

Missing values per column:
 order_id      0
customer      0
city          0
category      0
price         2
quantity      0
order_date    0
is_fraud      0
dtype: int64

Filled missing prices with median ($285.0). Remaining NaNs: 0


,order_id,customer,city,category,price,quantity,order_date,is_fraud
0,1001,Sara,Cairo,Electronics,500.0,1,2026-01-05,No
1,1002,Omar,Alexandria,Clothing,80.0,2,2026-01-06,No
2,1003,Lina,Cairo,Books,25.0,3,2026-01-06,No
3,1004,Youssef,Giza,Electronics,1200.0,1,2026-01-08,Yes
4,1005,Mona,Cairo,Clothing,285.0,2,2026-01-09,No
5,1006,Karim,Alexandria,Books,30.0,4,2026-01-10,No
6,1007,Nour,Cairo,Electronics,450.0,1,2026-01-11,No
7,1008,Tarek,Giza,Clothing,120.0,1,2026-01-12,No
8,1009,Hana,Cairo,Books,285.0,2,2026-01-14,No
9,1010,Ziad,Alexandria,Electronics,800.0,1,2026-01-15,No


### ✏️ Practice Exercise 2: Data Cleaning
Suppose you receive a tiny messy survey table `survey_df`:
1. Remove duplicate rows (`drop_duplicates()`).
2. Clean the `"country"` column using `.str.strip().str.upper()` so `"egypt"` and `" Egypt "` match.
3. Convert `"age"` to numeric with `pd.to_numeric(..., errors="coerce")` and fill any missing `NaN` ages with the **mean** age.

In [13]:
survey_df = pd.DataFrame({
    "user_id": [1, 2, 2, 3, 4],
    "country": [" egypt", "EGYPT ", "EGYPT ", "kenya", "Egypt"],
    "age":     ["25", "30", "30", "unknown", "35"]
})

# Remove duplicate rows (drop_duplicates()).
survey_df = survey_df.drop_duplicates()
print(survey_df)

   user_id country      age
0        1   egypt       25
1        2  EGYPT        30
3        3   kenya  unknown
4        4   Egypt       35


In [16]:
# Clean the "country" column using .str.strip().str.upper() so "egypt" and " Egypt " match.
survey_df["country"] = survey_df["country"].str.strip().str.upper()
print(survey_df)

   user_id country   age
0        1   EGYPT  25.0
1        2   EGYPT  30.0
3        3   KENYA  30.0
4        4   EGYPT  35.0


In [17]:
# Convert "age" to numeric with pd.to_numeric(..., errors="coerce") and fill any missing NaN ages with the mean age.

# Step 1: Convert 'age' to numeric, coercing invalid values to NaN
survey_df['age'] = pd.to_numeric(survey_df['age'], errors='coerce')

# Step 2: Calculate mean age ignoring NaN
mean_age = survey_df['age'].mean()

# Step 3: Fill NaN values with the mean age
survey_df['age'] = survey_df['age'].fillna(mean_age)

# Display the cleaned DataFrame
print(survey_df)


   user_id country   age
0        1   EGYPT  25.0
1        2   EGYPT  30.0
3        3   KENYA  30.0
4        4   EGYPT  35.0


<details>
<summary><strong>💡 Click to view solution</strong></summary>

```python
survey_clean = survey_df.drop_duplicates().copy()
survey_clean["country"] = survey_clean["country"].str.strip().str.upper()
survey_clean["age"] = pd.to_numeric(survey_clean["age"], errors="coerce")
survey_clean["age"] = survey_clean["age"].fillna(survey_clean["age"].mean())
survey_clean
```

</details>

---
## 6. Modifying Columns, Sorting & Simple Feature Engineering

### 🌍 Real-World Use Case
Raw tables rarely have the exact metric you want to plot or feed into an ML model:
- You have `price` and `quantity`, so you engineer **`revenue = price * quantity`** (feature engineering!).
- You encode `"Yes"`/`"No"` fraud labels into `1`/`0` using **`.map()`** so ML algorithms can understand them.
- You sort by `revenue` or call **`nlargest(3)`** to immediately see your top 3 biggest orders.

In [18]:
# 1. Create a new calculated column: revenue
df_clean["revenue"] = df_clean["price"] * df_clean["quantity"]

# 2. Map Yes/No text to 1/0 numeric target for ML
df_clean["fraud_flag"] = df_clean["is_fraud"].map({"No": 0, "Yes": 1})

# 3. Rename a column and drop one we no longer need
df_clean = df_clean.rename(columns={"customer": "customer_name"})

# 4. Sort and find top 3 highest-revenue orders
print("Top 3 orders by revenue:")
display(df_clean.nlargest(3, "revenue")[["order_id", "customer_name", "category", "revenue"]])

Top 3 orders by revenue:


,order_id,customer_name,category,revenue
3,1004,Youssef,Electronics,1200.0
9,1010,Ziad,Electronics,800.0
4,1005,Mona,Clothing,570.0


---
## 7. Grouping & Aggregating (`groupby` & `crosstab`)

### 🌍 Real-World Use Case
During EDA, stakeholders always ask **group-level questions**:
- *"Which **city** generates the highest total revenue?"*
- *"What is the **average order value** and **order count** for each product category?"*
- *"Is fraud happening more often in one category than another?"*

Pandas **`groupby()`** follows the **Split → Apply → Combine** pattern: it splits rows into groups (e.g., by `city`), applies an aggregation (`sum`, `mean`, `count`), and combines the results into a clean summary table.

In [19]:
# Total & average revenue per city
city_summary = (
    df_clean
    .groupby("city")["revenue"]
    .agg(["count", "sum", "mean"])
    .sort_values(by="sum", ascending=False)
)
print("Revenue summary by city:")
display(city_summary)

# Grouping by two columns with .reset_index() to keep a flat DataFrame
cat_city = (
    df_clean
    .groupby(["city", "category"])["revenue"]
    .sum()
    .reset_index()
)
print("Revenue by city & category:")
display(cat_city)

# Cross-tabulation: quick frequency table between two categorical columns
print("Order counts by Category vs. Fraud:")
display(pd.crosstab(df_clean["category"], df_clean["is_fraud"]))

Revenue summary by city:


,count,sum,mean
city,,,
Cairo,5,2165.0,433.0
Giza,2,1320.0,660.0
Alexandria,3,1080.0,360.0


Revenue by city & category:


,city,category,revenue
0,Alexandria,Books,120.0
1,Alexandria,Clothing,160.0
2,Alexandria,Electronics,800.0
3,Cairo,Books,645.0
4,Cairo,Clothing,570.0
5,Cairo,Electronics,950.0
6,Giza,Clothing,120.0
7,Giza,Electronics,1200.0


Order counts by Category vs. Fraud:


is_fraud,No,Yes
category,,
Books,3,0
Clothing,3,0
Electronics,3,1


### ✏️ Practice Exercise 3: `groupby` Insights
Using `df_clean`:
1. Group by `"category"` and compute the **total quantity sold** (`sum`) and **average price** (`mean`) for each category (hint: `.groupby("category").agg({"quantity": "sum", "price": "mean"})`).
2. Sort the result from highest total quantity to lowest.

In [20]:
# Group by "category" and compute the total quantity sold (sum) and average price (mean) for each category (hint: .groupby("category").agg({"quantity": "sum", "price": "mean"})).
new_df = df_clean.groupby("category").agg({"quantity": "sum", "price": "mean"})
print (new_df)

             quantity       price
category                         
Books               9  113.333333
Clothing            5  161.666667
Electronics         4  737.500000


In [22]:
# Sort the result from highest total quantity to lowest.
df_sorted = new_df.sort_values(by='quantity', ascending=False, ignore_index=True)
print(df_sorted)

   quantity       price
0         9  113.333333
1         5  161.666667
2         4  737.500000


<details>
<summary><strong>💡 Click to view solution</strong></summary>

```python
cat_stats = (
    df_clean
    .groupby("category")
    .agg({"quantity": "sum", "price": "mean"})
    .sort_values(by="quantity", ascending=False)
)
cat_stats
```

</details>

---
## 8. Combining Tables (`concat` & `merge`)

### 🌍 Real-World Use Case
Real company data lives in **multiple tables**:
- **`pd.concat()` (stacking):** Combining *January orders* and *February orders* (same columns) into one big table.
- **`pd.merge()` (SQL-style JOIN):** Enriching your `orders` table with a separate `customers` table (adding each customer's `loyalty_tier` or `signup_year` by matching `customer_name`). This is how analysts assemble a single **ML feature table**!

In [23]:
# 1) pd.concat: stack two tables with matching columns
extra_orders = pd.DataFrame({
    "order_id": [1011, 1012],
    "customer_name": ["Sara", "Omar"],
    "city": ["Cairo", "Alexandria"],
    "category": ["Books", "Clothing"],
    "price": [40.0, 95.0],
    "quantity": [2, 1],
    "revenue": [80.0, 95.0]
})

combined = pd.concat([df_clean[["order_id", "customer_name", "city", "category", "price", "quantity", "revenue"]],
                      extra_orders], ignore_index=True)
print("Combined rows:", len(combined))

# 2) pd.merge: join orders with a customer loyalty table (left join keeps all orders)
loyalty_table = pd.DataFrame({
    "customer_name": ["Sara", "Omar", "Youssef", "Mona"],
    "loyalty_tier": ["Gold", "Silver", "Gold", "Bronze"]
})

enriched = pd.merge(combined, loyalty_table, on="customer_name", how="left")
print("\nOrders enriched with loyalty_tier (notice NaN for customers not in loyalty_table):")
display(enriched[["order_id", "customer_name", "revenue", "loyalty_tier"]].head(6))

Combined rows: 12

Orders enriched with loyalty_tier (notice NaN for customers not in loyalty_table):


,order_id,customer_name,revenue,loyalty_tier
0,1001,Sara,500.0,Gold
1,1002,Omar,160.0,Silver
2,1003,Lina,75.0,NaN
3,1004,Youssef,1200.0,Gold
4,1005,Mona,570.0,Bronze
5,1006,Karim,120.0,NaN


---
## 9. Date & Time Essentials

### 🌍 Real-World Use Case
In `df_clean`, `order_date` is currently plain text (`object`). Once you convert it with **`pd.to_datetime()`**, you unlock the **`.dt`** accessor to extract `month`, `day_name()`, or `dayofweek` — helping you discover whether customers shop more on **weekends** or during **specific months**!

In [24]:
df_clean["order_date"] = pd.to_datetime(df_clean["order_date"])

# Extract useful time features for EDA
df_clean["order_month"] = df_clean["order_date"].dt.month
df_clean["day_of_week"] = df_clean["order_date"].dt.day_name()

print("Orders by day of week:")
print(df_clean["day_of_week"].value_counts())

df_clean[["order_id", "order_date", "order_month", "day_of_week"]].head(4)

Orders by day of week:
day_of_week
Monday       2
Tuesday      2
Thursday     2
Friday       1
Saturday     1
Sunday       1
Wednesday    1
Name: count, dtype: int64


,order_id,order_date,order_month,day_of_week
0,1001,2026-01-05,1,Monday
1,1002,2026-01-06,1,Tuesday
2,1003,2026-01-06,1,Tuesday
3,1004,2026-01-08,1,Thursday


---
## 10. Capstone Mini-EDA Lab & Summary Cheat Sheet

### 🌍 Scenario: Analyzing a Messy Course-Enrollment Dataset
You are given a raw enrollment table `enrollments_raw` with several classic real-world problems:
- A duplicate row
- Inconsistent casing/spaces in `"track"` (`" data"`, `"DATA"`, `"Data"`)
- A missing `"fee"` (`NaN`)
- Text dates (`"signup_date"`)

**Your Tasks:**
1. Make a copy `enrollments`, remove duplicates, and standardize `"track"` using `.str.strip().str.title()`.
2. Fill missing `"fee"` values with the **median** fee.
3. Convert `"signup_date"` to datetime and extract `"signup_month"`.
4. Use `value_counts()` on `"track"` to check which track is most popular.
5. Use `groupby("track")["fee"].agg(["count", "sum", "mean"])` to summarize revenue per track!

In [27]:
enrollments_raw = pd.DataFrame({
    "student_id": [101, 102, 103, 103, 104, 105, 106],
    "track":      [" data", "Web", "DATA", "DATA", "Ai", "web ", "Data"],
    "fee":        [200.0, 150.0, 200.0, 200.0, np.nan, 150.0, 220.0],
    "signup_date":["2026-01-10", "2026-01-15", "2026-02-02", "2026-02-02", "2026-02-14", "2026-02-20", "2026-03-01"]
})

# Make a copy enrollments, remove duplicates, and standardize "track" using .str.strip().str.title()
enrollments = enrollments_raw.drop_duplicates()
enrollments["track"] = enrollments["track"].str.strip().str.title()
print(enrollments)

   student_id track    fee signup_date
0         101  Data  200.0  2026-01-10
1         102   Web  150.0  2026-01-15
2         103  Data  200.0  2026-02-02
4         104    Ai    NaN  2026-02-14
5         105   Web  150.0  2026-02-20
6         106  Data  220.0  2026-03-01


In [28]:
# Fill missing "fee" values with the median fee.

# Calculate median of 'fee' ignoring NaN
median_fee = enrollments["fee"].median()

# Fill NaN values in 'fee' with the median
enrollments['fee'] = enrollments['fee'].fillna(median_fee)

# Display updated DataFrame
print(enrollments)

   student_id track    fee signup_date
0         101  Data  200.0  2026-01-10
1         102   Web  150.0  2026-01-15
2         103  Data  200.0  2026-02-02
4         104    Ai  200.0  2026-02-14
5         105   Web  150.0  2026-02-20
6         106  Data  220.0  2026-03-01


In [30]:
# Convert "signup_date" to datetime and extract "signup_month".

enrollments['signup_date'] = pd.to_datetime(enrollments['signup_date'], errors='coerce')
print(enrollments)
enrollments.info()

   student_id track    fee signup_date
0         101  Data  200.0  2026-01-10
1         102   Web  150.0  2026-01-15
2         103  Data  200.0  2026-02-02
4         104    Ai  200.0  2026-02-14
5         105   Web  150.0  2026-02-20
6         106  Data  220.0  2026-03-01
<class 'pandas.DataFrame'>
Index: 6 entries, 0 to 6
Data columns (total 4 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   student_id   6 non-null      int64         
 1   track        6 non-null      str           
 2   fee          6 non-null      float64       
 3   signup_date  6 non-null      datetime64[us]
dtypes: datetime64[us](1), float64(1), int64(1), str(1)
memory usage: 261.0 bytes


In [31]:
# Use value_counts() on "track" to check which track is most popular.
track_count = enrollments['track'].value_counts()
print(track_count)

track
Data    3
Web     2
Ai      1
Name: count, dtype: int64


In [32]:
# Use groupby("track")["fee"].agg(["count", "sum", "mean"]) to summarize revenue per track

res1 = enrollments.groupby("track")["fee"].agg(["count", "sum", "mean"])
print(res1)

       count    sum        mean
track                          
Ai         1  200.0  200.000000
Data       3  620.0  206.666667
Web        2  300.0  150.000000


<details>
<summary><strong>💡 Click to view solution</strong></summary>

```python
enrollments = enrollments_raw.drop_duplicates().copy()

# 1. Clean text
enrollments["track"] = enrollments["track"].str.strip().str.title()

# 2. Fill missing fees with median
enrollments["fee"] = enrollments["fee"].fillna(enrollments["fee"].median())

# 3. Parse dates & extract month
enrollments["signup_date"] = pd.to_datetime(enrollments["signup_date"])
enrollments["signup_month"] = enrollments["signup_date"].dt.month

# 4. Track popularity
print("Track counts:\n", enrollments["track"].value_counts())

# 5. Revenue summary by track
track_summary = enrollments.groupby("track")["fee"].agg(["count", "sum", "mean"])
print("\nRevenue by track:")
track_summary
```

</details>

---
### 📋 Pandas Quick-Reference Cheat Sheet

| Task | Essential Syntax |
| :--- | :--- |
| **Load / Save** | `pd.read_csv("file.csv")`, `df.to_csv("out.csv", index=False)` |
| **First-Look EDA** | `df.head()`, `df.shape`, `df.info()`, `df.describe()`, `df["col"].value_counts(normalize=True)` |
| **Select & Filter** | `df[["a", "b"]]`, `df.loc[rows, cols]`, `df.iloc[0:5, 0:2]`, `df[df["col"].isin([...])]` |
| **Clean Missing/Dups** | `df.isna().sum()`, `df.dropna()`, `df.fillna(val)`, `df.duplicated()`, `df.drop_duplicates()` |
| **Types & Strings** | `pd.to_numeric(s, errors="coerce")`, `s.astype(float)`, `s.str.strip().str.lower()` |
| **Engineer & Sort** | `df["new"] = ...`, `df["col"].map({...})`, `df.sort_values("col", ascending=False)`, `df.nlargest(5, "col")` |
| **Group & Join** | `df.groupby("g")["v"].agg(["count", "mean", "sum"])`, `pd.crosstab()`, `pd.concat()`, `pd.merge()` |
| **Dates** | `pd.to_datetime(df["date"])`, `.dt.year`, `.dt.month`, `.dt.day_name()` |

---
### 🚀 What's Next?
Numbers in tables tell half the story — **charts** reveal patterns, outliers, and distributions in a single glance!  
In the next notebook (**`03_matplotlib.ipynb`**), we'll turn our cleaned Pandas & NumPy data into clear, customizable visualizations.